# Local SSD RAID0 speed test on Verily Workbench Dataproc

What this notebook does, in one sentence: it makes a Dataproc cluster, glues its local SSDs
into one fast volume (RAID0), and measures how fast you can really write and read it.

What we measured (so you know what to expect):

| what | speed (per worker) |
|---|---|
| default GCS bucket download | ~0.14 GiB/s |
| 4 local SSDs, plain read | ~1.4 GiB/s |
| 16 local SSDs, big block read | ~3.5 GiB/s |
| 16 local SSDs, 8 reads at once | ~6 GiB/s (the official SCSI maximum) |

Cost: local SSDs are $0.08 per GiB per month (375 GiB disk = $30/month = ~$0.04/hour).
The 2-worker test cluster below costs about $5/hour total. Delete it when done (last step).

What you need before starting:
- the `wb` (workbench-cli) tool, logged in, with your workspace active;
- a Dataproc cluster created by this cluster recipe (step 1);
- the JupyterLab that runs ON the cluster manager (step 3).


## Step 1 - Create the cluster (run in your TERMINAL, not here)

```bash
wb resource create dataproc-cluster \
  --id=ssd-raid0 \
  --region=us-central1 \
  --quiet --format=JSON \
  --num-workers=2 \
  --worker-machine-type=n2-standard-32 \
  --worker-boot-disk-size=100 \
  --worker-num-local-ssds=16
```

The parameters:
- `--id` - name of the cluster. Use the same name in step 3 below.
- `--num-workers=2` - two worker machines. Each one gets its own local SSDs.
- `--worker-machine-type` - CPU/RAM size. n2-standard-32 = 32 CPUs, 128 GB RAM. Any
  big N1/N2 type works; the disk speed only depends on the number of local SSDs.
- `--worker-num-local-ssds=16` - 16 fast disks of 375 GiB each, per worker. This is the
  whole point. The speed cap goes UP with the number of disks (4-8 disks: 1,560 MiB/s,
  16 disks: 6,240 MiB/s, 24 disks: 9,360 MiB/s read, per the Google local-SSD table).
- `--worker-boot-disk-size=100` - normal boot disk, nothing special.

Two warnings:
1. The create command often prints an error (401) EVEN THOUGH it worked. Never run it
   twice right away. Check first: `wb resource list`.
2. Creation takes about 5-15 minutes and the command says nothing while waiting.


## Step 2 - Wait until it runs (terminal)

```bash
wb resource describe --id=ssd-raid0 --format=JSON | grep -E '"status"|"proxyUri"'
```

Wait for `"status": "RUNNING"` and copy the `proxyUri` link. That link opens JupyterLab
of the cluster. Note: the create/delete commands print nothing for 10-20 minutes. That is
normal, they are working, not frozen.

## Step 3 - Open JupyterLab, start THIS notebook

Open the proxyUri link. Create a new notebook and pick the **Python 3** kernel
(NOT "PySpark" - the PySpark kernel starts a background Spark app that causes noise).

Set the cluster name here - everything below uses it:

In [ ]:
CLUSTER = "ssd-raid0"   # the --id you used in step 1
print("using cluster:", CLUSTER)

## Step 4 - Start the result printer (run first, keep it alive)

The slow tests run in the background ON THE WORKER MACHINES and send their progress lines
to this cell while they work. Start this and leave it alone (the cell stays busy - that is
the point). If you stop it, you will not see the test results.

In [ ]:
import threading, http.server, socketserver, time
try:
    LOGS
    print("listener already up; LOGS:", list(LOGS))
except NameError:
    LOGS = {}
    class HH(http.server.BaseHTTPRequestHandler):
        def do_POST(self):
            n = int(self.headers.get("Content-Length", 0))
            host = self.path.strip("/") or "unknown"
            LOGS[host] = self.rfile.read(n).decode(errors="replace")
            print(f"\n===== {host} @ {time.strftime('%H:%M:%S')} =====\n...{LOGS[host][-500:]}", flush=True)
            self.send_response(200); self.end_headers()
        def log_message(self, *a): pass
    class SRV(socketserver.ThreadingTCPServer):
        allow_reuse_address = True; daemon_threads = True
    threading.Thread(target=SRV(("0.0.0.0", 18888), HH).serve_forever, daemon=True).start()
    print("listener started fresh")


## Step 5 - Clean up stray Spark apps (optional, silences noise)

Jupyter sometimes starts its own background Spark app. This finds it in the YARN scheduler
and kills it, so workers stay free for our tests.

In [ ]:
import subprocess, json
r = subprocess.run(["sudo","curl","-s","-m10","http://localhost:8088/ws/v1/cluster/apps"], capture_output=True, text=True, timeout=30)
for a in json.loads(r.stdout)["apps"]["app"]:
    if a["state"] in ("ACCEPTED", "RUNNING"):
        k = subprocess.run(f"sudo -u yarn yarn application -kill {a['id']}", shell=True, capture_output=True, text=True, timeout=60)
        print("KILLED", a["id"], "->", (k.stdout or k.stderr).strip()[-80:], flush=True)
    else:
        print(a["id"], a["state"], a["finalStatus"])


## Step 6 - Look at the disks (one quick check)

This prints, for each worker: the disk names and the machine type. `sd*` names mean the
disks came as SCSI (normal for Dataproc right now); `nvme*` names would mean NVMe. This
matters because the speed cap depends on the interface (see the README notes).

In [ ]:
import subprocess
JOB = '''
import socket, subprocess, os
def probe():
    h = os.uname().nodename
    ls = subprocess.run("lsblk -d -o NAME,TRAN,MODEL,SIZE | grep 375G", shell=True, capture_output=True, text=True).stdout.replace("\\n","; ")
    nv = subprocess.run("ls /dev/nvme*n1 2>/dev/null || echo no-nvme-disks", shell=True, capture_output=True, text=True).stdout.strip()
    mt = subprocess.run('curl -s -H "Metadata-Flavor: Google" http://metadata.google.internal/computeMetadata/v1/instance/machine-type', shell=True, capture_output=True, text=True).stdout
    return "IFACE " + h + " | " + mt.split("/")[-1] + " | 375G-devs: " + ls.strip() + " | " + nv
if __name__ == "__main__":
    from pyspark.sql import SparkSession
    sp = SparkSession.builder.appName("iface").getOrCreate()
    for x in sorted(set(sp.sparkContext.parallelize(range(8), 8).map(lambda _: probe()).collect())):
        print(x, flush=True)
    sp.stop()
'''
open("/tmp/job_iface.py", "w").write(JOB)
p = subprocess.Popen(["spark-submit","--master","yarn","--deploy-mode","client",
   "--conf","spark.executor.instances=2","--conf","spark.executor.memory=1g","/tmp/job_iface.py"],
   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    if line.startswith("IFACE"): print(line.rstrip()[:250], flush=True)
p.wait()


## Step 7 - Glue the disks together (RAID0 stripe)

What this cell does on each worker (through the worker's own Docker, as root):
1. stops the Hadoop/Yarn services that hold the disks,
2. finds every 375-GiB disk by SIZE (so the boot disk can never be touched),
3. glues all 16 into one device with `dmsetup` (stripe = classic RAID0),
4. makes one big ext4 filesystem, mounts it at `/mnt/raid` (~5.9 TiB),
5. fixes the Hadoop folder owners and restarts the services.

It runs once; running it again says ALREADY and does nothing. Expect
`LAUNCH_RESULT ... rc=0` for both workers, then look at step 8.

In [ ]:
import subprocess
V8 = r"""#!/bin/bash
export PATH=/usr/sbin:/usr/bin:/sbin:/bin
exec >> /tmp/prep-v8.log 2>&1
set -x
shopt -s nullglob
H=$(hostname); MGR="${H%%-w-*}-m.$(hostname -d)"
post() { curl -m5 -s -X POST --data-binary @/tmp/prep-v8.log "http://$MGR:18888/$H" || true; }
date; echo "V8 START $H"; post
exec 9>/tmp/strip.lock; flock -n 9 || { echo ALREADY_RUNNING; exit 0; }
LDIRS=$(cut -f2 -d' ' /proc/mounts | grep "^/mnt/" | tr '\n' ' ')
echo "LDIRS=$LDIRS"; post
DISKS=""
for b in /sys/block/sd* /sys/block/nvme*n1; do S=$(cat $b/size 2>/dev/null); if [ -n "$S" ] && [ "$S" -gt 700000000 ]; then DISKS="$DISKS /dev/$(basename $b)"; fi; done
N=$(echo $DISKS | wc -w); echo "DISKS=$DISKS N=$N"
[ "$N" -lt 16 ] && { echo WANT_16_GOT_$N; post; exit 1; }
SZ=$(for d in $DISKS; do cat /sys/block/$(basename $d)/size; done | sort -n | head -1)
LEN=$((SZ*N)); echo "SZ=$SZ LEN=$LEN"; post
systemctl stop hadoop-yarn-nodemanager hadoop-hdfs-datanode 2>&1; sleep 3
for m in $LDIRS; do umount -l $m; done
sleep 2
ARGS=""; for d in $DISKS; do ARGS="$ARGS $d 0"; done
OK=""
for T in striped stripe; do
  dmsetup create ssdraid --table "0 $LEN $T $N 256 $ARGS" && { echo DM_OK=$T; OK=1; break; }
  dmsetup remove ssdraid 2>/dev/null
done
post
if [ -n "$OK" ] && dmsetup info ssdraid >/dev/null 2>&1; then
  echo MKFS_BEGIN; post
  mkfs.ext4 -F -q /dev/mapper/ssdraid && mkdir -p /mnt/raid && mount /dev/mapper/ssdraid /mnt/raid && chmod 1777 /mnt/raid && echo ALLDONE_6T
  df -h /mnt/raid
else
  echo DM_FAIL
fi
for m in $LDIRS; do
  mkdir -p $m/hadoop/dfs/data $m/hadoop/yarn/nm-local-dir
  chown -R hdfs:hadoop $m/hadoop/dfs
  chown -R yarn:yarn $m/hadoop/yarn
  chown root:root $m; chmod 0755 $m
done
echo DIRS_FIXED; post
systemctl restart hadoop-hdfs-datanode 2>&1 || true
systemctl restart hadoop-yarn-nodemanager 2>&1 || true
sleep 15
echo "DN=$(systemctl is-active hadoop-hdfs-datanode 2>&1) NM=$(systemctl is-active hadoop-yarn-nodemanager 2>&1)"
echo V8_DONE; date; post
"""
JOB = '''
import socket, subprocess
V8 = open("/tmp/strip-v8.sh").read()
def launch():
    h = socket.gethostname(); name = "strip-" + h.split(".")[0]
    if subprocess.run(["bash","-c","docker ps -a --format '{{.Names}}' | grep -qx " + name],
                      capture_output=True).returncode == 0:
        return (h, "ALREADY")
    open("/tmp/strip-v8.sh", "w").write(V8)
    subprocess.run("tar -c --files-from /dev/null | docker import - nullimg", shell=True, capture_output=True)
    r = subprocess.run("docker run -d --name " + name + " --privileged --pid=host --uts=host --network=host --ipc=host "
                       "-v /:/host -v /lib64:/lib64 -v /usr:/usr -v /tmp:/tmp nullimg "
                       "/host/bin/bash -c '/usr/bin/nsenter -t 1 -m -- /bin/bash /tmp/strip-v8.sh'",
                       shell=True, capture_output=True, text=True)
    return (h, "rc=%s %s" % (r.returncode, r.stderr[-80:]))
if __name__ == "__main__":
    from pyspark.sql import SparkSession
    sp = SparkSession.builder.appName("strip-launch8").getOrCreate()
    for x in sorted(set(sp.sparkContext.parallelize(range(16), 16).map(lambda _: launch()).collect())):
        print("LAUNCH_RESULT:", x, flush=True)
    sp.stop()
'''
open("/tmp/strip-v8.sh", "w").write(V8)
open("/tmp/job_strip8.py", "w").write(JOB)
cmd = ["spark-submit", "--master", "yarn", "--deploy-mode", "client",
       "--conf", "spark.executor.instances=2",
       "--conf", "spark.excludeOnFailure.enabled=false",
       "--conf", "spark.yarn.executor.launch.excludeOnFailure.enabled=false",
       "--conf", "spark.executor.maxNumFailures=1000",
       "--conf", "spark.executor.memory=1g", "/tmp/job_strip8.py"]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    if line[:3] != "26/" and not line.startswith("\tat "): print(line.rstrip()[:300], flush=True)
print("rc:", p.wait(), flush=True)


## Step 8 - Check that the stripe is healthy

First cell: fetches the strip log + disk size from each worker. You want to see
`ALLDONE_6T`, a `5.9T` line for `/mnt/raid`, and `DN=active NM=active`.
Second cell: asks the cluster scheduler if both workers are healthy (`RUNNING | OK`).

In [ ]:
import subprocess
JOB = '''
import socket, subprocess, os
def probe():
    h = socket.gethostname().split(".")[0]
    def sh(c):
        r = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=30)
        return ((r.stdout or "") + (r.stderr or ""))[-1200:]
    log = open("/tmp/prep-v8.log").read()[-1800:] if os.path.exists("/tmp/prep-v8.log") else "MISSING prep-v8.log (script never ran)"
    docker = sh("docker ps -a --format '{{.Names}} {{.Status}}' | grep strip || echo no-strip-container")
    df = sh("df -h /mnt/raid | tail -1")
    return "\\n##### " + h + " #####\\n" + log + "\\nDOCKER: " + docker + "\\nDF: " + df
if __name__ == "__main__":
    from pyspark.sql import SparkSession
    sp = SparkSession.builder.appName("fetch").getOrCreate()
    out = sp.sparkContext.parallelize(range(16), 16).map(lambda _: probe()).collect()
    for x in sorted(set(out)): print(x, flush=True)
    sp.stop()
'''
open("/tmp/job_fetch.py", "w").write(JOB)
p = subprocess.Popen(["spark-submit", "--master", "yarn", "--deploy-mode", "client",
                      "--conf", "spark.executor.instances=2", "--conf", "spark.executor.memory=1g",
                      "/tmp/job_fetch.py"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    if line[:3] != "26/" and not line.startswith("\tat "): print(line.rstrip()[:300], flush=True)
print("rc:", p.wait(), flush=True)


In [ ]:
import subprocess, json
r = subprocess.run(["sudo","curl","-s","-m10","http://localhost:8088/ws/v1/cluster/nodes"], capture_output=True, text=True, timeout=30)
for x in json.loads(r.stdout)["nodes"]["node"]:
    print("NODE:", x["nodeHostName"].split(".")[0], "|", x["state"], "|", x["healthReport"][:100] or "OK")


## Step 9 - Write test: 40 GiB per worker

Writes 40 GiB of zeros straight to the disks (bypassing the Linux page cache, so the
number is real disk speed, not RAM). Progress lines go to the step-4 cell.
Watch for `W_DONE <worker> <seconds> <GiB/s>`. With 16 disks expect ~3 GiB/s.

In [ ]:
import subprocess, random
FR = r"""#!/bin/bash
export PATH=/usr/sbin:/usr/bin:/sbin:/bin
H=$(hostname); MGR="${H%%-w-*}-m.$(hostname -d)"
post() { curl -m5 -s -X POST --data-binary "$1" "http://$MGR:18888/w-$H" || true; }
exec 9>/tmp/w.lock; flock -n 9 || { post "W_SKIPLOCK $H"; exit 0; }
F=/mnt/raid/benchfile; T=$((40*1024*1024*1024))
if [ "$(stat -c %s "$F" 2>/dev/null || echo 0)" = "$T" ]; then post "W_ALREADY $H"; exit 0; fi
rm -f "$F"; post "W_START $H"; t0=$(date +%s)
dd if=/dev/zero of="$F" bs=4M count=10240 oflag=direct status=noxfer 2>/tmp/w.err & pid=$!
while kill -0 $pid 2>/dev/null; do sleep 10
  wb=$(awk '/^write_bytes/{print $2}' /proc/$pid/io 2>/dev/null); e=$(( $(date +%s)-t0 ))
  [ -n "$wb" ] && post "W $H ${e}s $(awk "BEGIN{printf \"%.2f\", $wb/1073741824}") GiB ~$(awk "BEGIN{printf \"%.2f\", $wb/1073741824/(($e)+1)}") GiB/s"
done; wait $pid; e=$(( $(date +%s)-t0 ))
post "W_DONE $H ${e}s $(awk "BEGIN{printf \"%.2f\", 40/$e}") GiB/s"
"""
JOB = '''
import socket, subprocess
FR = open("/tmp/w.sh").read()
def launch():
    h = socket.gethostname().split(".")[0]
    open("/tmp/w.sh", "w").write(FR)
    r = subprocess.run("docker run -d --name w-" + SUFFIX + "-" + h + " --privileged --pid=host --uts=host --network=host --ipc=host "
                       "-v /:/host -v /lib64:/lib64 -v /lib:/lib -v /usr:/usr -v /tmp:/tmp nullimg "
                       "/host/bin/bash -c '/usr/bin/nsenter -t 1 -m -- /bin/bash /tmp/w.sh'",
                       shell=True, capture_output=True, text=True)
    return "WOK " + h + " rc=" + str(r.returncode)
if __name__ == "__main__":
    import os
    SUFFIX = os.environ["SFX"]
    from pyspark.sql import SparkSession
    sp = SparkSession.builder.appName("w-launch").getOrCreate()
    for x in sorted(set(sp.sparkContext.parallelize(range(16), 16).map(lambda _: launch()).collect())):
        print("WLAUNCH:", x, flush=True)
    sp.stop()
'''
open("/tmp/w.sh", "w").write(FR); open("/tmp/job_w.py", "w").write(JOB)
import os
env = dict(os.environ, SFX="%04x" % random.randrange(65536))
p = subprocess.Popen(["spark-submit","--master","yarn","--deploy-mode","client",
   "--conf","spark.executor.instances=2","--conf","spark.executor.memory=1g","/tmp/job_w.py"],
   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in p.stdout:
    if line.startswith("WLAUNCH:"): print(line.rstrip()[:120], flush=True)
p.wait(); print("watch LISTENER cell for W_DONE x2", flush=True)


## Step 10 - Read tests

**10a. Same file, 3 times in a row.** Reads the 40 GiB back three times with 4 MiB
requests. This separates a short speed "burst" from the real steady speed. Look for
`R_ALLDONE <worker> total <s>s passes: a b c`. Expect ~3.4 GiB/s, same on all passes.

In [ ]:
import subprocess, random
FR = r"""#!/bin/bash
export PATH=/usr/sbin:/usr/bin:/sbin:/bin
H=$(hostname); MGR="${H%%-w-*}-m.$(hostname -d)"
post() { curl -m5 -s -X POST --data-binary "$1" "http://$MGR:18888/r-$H" || true; }
exec 9>/tmp/r.lock; flock -n 9 || { post "R_SKIPLOCK $H"; exit 0; }
F=/mnt/raid/benchfile
if [ "$(stat -c %s "$F" 2>/dev/null || echo 0)" != "$((40*1024*1024*1024))" ]; then post "R_NOFILE $H"; exit 0; fi
SUM=0; ALL=""
for p in 1 2 3; do
  post "R_P${p}_START $H"; t0=$(date +%s)
  dd if="$F" of=/dev/null bs=4M count=10240 iflag=direct status=noxfer 2>/tmp/r.err & pid=$!
  ( while sleep 5; do kill -0 $pid 2>/dev/null || break
      rb=$(awk '/^read_bytes/{print $2}' /proc/$pid/io 2>/dev/null); e=$(( $(date +%s)-t0 ))
      [ -n "$rb" ] && post "R_P${p} $H ${e}s $(awk "BEGIN{printf \"%.2f\", $rb/1073741824}") GiB ~$(awk "BEGIN{printf \"%.2f\", $rb/1073741824/(($e)+1)}") GiB/s"
    done ) & mon=$!
  wait $pid; kill $mon 2>/dev/null; e=$(( $(date +%s)-t0 )); r=$(awk "BEGIN{printf \"%.2f\", 40/($e)}")
  post "R_P${p}DONE $H ${e}s ${r} GiB/s"; SUM=$((SUM+e)); ALL="$ALL ${r}"
done
post "R_ALLDONE $H total ${SUM}s passes:$ALL"
"""
JOB = '''
import socket, subprocess
FR = open("/tmp/r.sh").read()
def launch():
    h = socket.gethostname().split(".")[0]
    open("/tmp/r.sh", "w").write(FR)
    r = subprocess.run("docker run -d --name r-" + SUFFIX + "-" + h + " --privileged --pid=host --uts=host --network=host --ipc=host "
                       "-v /:/host -v /lib64:/lib64 -v /lib:/lib -v /usr:/usr -v /tmp:/tmp nullimg "
                       "/host/bin/bash -c '/usr/bin/nsenter -t 1 -m -- /bin/bash /tmp/r.sh'",
                       shell=True, capture_output=True, text=True)
    return "ROK " + h + " rc=" + str(r.returncode)
if __name__ == "__main__":
    import os
    SUFFIX = os.environ["SFX"]
    from pyspark.sql import SparkSession
    sp = SparkSession.builder.appName("r-launch").getOrCreate()
    for x in sorted(set(sp.sparkContext.parallelize(range(16), 16).map(lambda _: launch()).collect())):
        print("RLAUNCH:", x, flush=True)
    sp.stop()
'''
open("/tmp/r.sh", "w").write(FR); open("/tmp/job_r.py", "w").write(JOB)
import os
env = dict(os.environ, SFX="%04x" % random.randrange(65536))
p = subprocess.Popen(["spark-submit","--master","yarn","--deploy-mode","client",
   "--conf","spark.executor.instances=2","--conf","spark.executor.memory=1g","/tmp/job_r.py"],
   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in p.stdout:
    if line.startswith("RLAUNCH:"): print(line.rstrip()[:120], flush=True)
p.wait(); print("watch LISTENER cell for R_ALLDONE x2", flush=True)


**10b. 8 readers at once (deep queue).** One reader waits for every request to finish
before the next one, which leaves the 16 disks mostly idle. Eight readers keep ~128
requests in flight, which is what Google's speed table assumes. This is also the shape
that kvikio (storage->GPU streaming) uses. Expect ~6 GiB/s - the official 16-disk SCSI
maximum. If you see this number, your GPU reader can realistically eat 6 GiB/s per node.

In [ ]:
import subprocess, random
FR = r"""#!/bin/bash
export PATH=/usr/sbin:/usr/bin:/sbin:/bin
H=$(hostname); MGR="${H%%-w-*}-m.$(hostname -d)"
post() { curl -m5 -s -X POST --data-binary "$1" "http://$MGR:18888/r2-$H" || true; }
exec 9>/tmp/r2.lock; flock -n 9 || { post "R2_SKIPLOCK $H"; exit 0; }
F=/mnt/raid/benchfile
if [ "$(stat -c %s "$F" 2>/dev/null || echo 0)" != "$((40*1024*1024*1024))" ]; then post "R2_NOFILE $H"; exit 0; fi
post "R2_START $H 8 streams bs=4M"; t0=$(date +%s)
PIDS=""
for i in 0 1 2 3 4 5 6 7; do
  dd if="$F" of=/dev/null bs=4M count=1280 skip=$((i*1280)) iflag=direct status=noxfer 2>/dev/null &
  PIDS="$PIDS $!"
done
( while sleep 5; do
      TOT=0
      for pid in $PIDS; do
        rb=$(awk '/^read_bytes/{print $2}' /proc/$pid/io 2>/dev/null); TOT=$((TOT + ${rb:-0}))
      done
      e=$(( $(date +%s)-t0 ))
      post "R2 $H ${e}s $(awk "BEGIN{printf \"%.2f\", $TOT/1073741824}") GiB ~$(awk "BEGIN{printf \"%.2f\", $TOT/1073741824/(($e)+1)}") GiB/s"
    done ) & mon=$!
for pid in $PIDS; do wait $pid; done
kill $mon 2>/dev/null; e=$(( $(date +%s)-t0 ))
post "R2_DONE $H ${e}s $(awk "BEGIN{printf \"%.2f\", 40/($e)}") GiB/s aggregate-8-streams"
"""
JOB = '''
import socket, subprocess
FR = open("/tmp/r2.sh").read()
def launch():
    h = socket.gethostname().split(".")[0]
    open("/tmp/r2.sh", "w").write(FR)
    r = subprocess.run("docker run -d --name r2-" + SUFFIX + "-" + h + " --privileged --pid=host --uts=host --network=host --ipc=host "
                       "-v /:/host -v /lib64:/lib64 -v /lib:/lib -v /usr:/usr -v /tmp:/tmp nullimg "
                       "/host/bin/bash -c '/usr/bin/nsenter -t 1 -m -- /bin/bash /tmp/r2.sh'",
                       shell=True, capture_output=True, text=True)
    return "R2OK " + h + " rc=" + str(r.returncode)
if __name__ == "__main__":
    import os
    SUFFIX = os.environ["SFX"]
    from pyspark.sql import SparkSession
    sp = SparkSession.builder.appName("r2-launch").getOrCreate()
    for x in sorted(set(sp.sparkContext.parallelize(range(16), 16).map(lambda _: launch()).collect())):
        print("R2LAUNCH:", x, flush=True)
    sp.stop()
'''
open("/tmp/r2.sh", "w").write(FR); open("/tmp/job_r2.py", "w").write(JOB)
import os
env = dict(os.environ, SFX="%04x" % random.randrange(65536))
p = subprocess.Popen(["spark-submit","--master","yarn","--deploy-mode","client",
   "--conf","spark.executor.instances=2","--conf","spark.executor.memory=1g","/tmp/job_r2.py"],
   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in p.stdout:
    if line.startswith("R2LAUNCH:"): print(line.rstrip()[:120], flush=True)
p.wait(); print("watch LISTENER cell for R2_DONE x2", flush=True)


## Step 11 - Delete everything (TERMINAL - do this, it is real money)

```bash
wb resource delete --id=ssd-raid0 --quiet
wb resource list | grep ssd     # must print nothing
```

Deleting also prints nothing for 10-20 minutes, then it is gone. If you leave the cluster
"just for tonight" it bills roughly $5/hour.

## What the numbers mean for a GPU streaming design

- One worker with 16 local SSDs gives ~6 GiB/s of steady raw reads when the reader keeps
  many requests in flight. A single slow reader sees only ~1.2-3.5 GiB/s - same disks,
  same machine, only the queue depth differs.
- The GPU slot is not the limit: a T4's PCIe slot passes ~11 GiB/s from RAM to GPU,
  comfortably above the 6 GiB/s the disks can feed it.
- The 16-disk trick only works on the plain CPU workers and on N1+GPU machines (N1 with
  T4/V100 accepts 16+ local SSDs). The L4 "G2" machines accept at most 8 local SSDs
  (~2.5 GiB/s), and A100 "A2" also stop at 8. If you need GPU + big local IO on one
  box, N1+T4 is the only cheap combination that can attach 16-24 local SSDs.
- NVMe interface would raise the per-disk-count caps further, but Terra's Workspace
  Manager currently drops the `localSsdInterface` argument (one missing line upstream,
  we filed it). Until that is fixed, everything here is the SCSI path.
- Prices (us-central1, on demand): machine ~$1.6/h per worker, local SSDs $0.08/GiB/month
  ($30/month per 375 GiB disk, ~$0.66/h for 16 of them). 24 disks instead of 16 buys the
  9,360 MiB/s row for +$0.33/h per worker.
